# Fine-tunning on Figure 3D MethylBERT data with Refined MethylBERT

This tutorial demonstrates:
1. **How to perform fine-tunning on the prepared data and save results to the local directory**.
---

### Step 0: Import needed classes and set paths

In [29]:
import os
from methyldl.modelling.classifiers.methylbert import MethylVocab, MethylBertFinetuneDataset
import pandas as pd
from methyldl.data.sequencing.genome import generate_kmer_str_with_overlap
import numpy as np

In [30]:
from methyldl.modelling.classifiers.methylbert import MethylBert, default_methylbert_config

In [31]:
data_path = '/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_withoutpooling_hg38_mincpg_4_minlen_10/'

In [32]:
data_prefix = data_path[data_path.find("mincpg") : -1]

In [33]:
dmr_label_column = "dmr_ctype_label"

In [34]:
dmrs = pd.DataFrame(
    pd.concat(
        [
            pd.read_parquet(data_path + f"/{split}.parquet")
            for split in ["train", "valid", "test"]
        ]
    )[dmr_label_column].unique()
)
dmrs.columns = [dmr_label_column]

In [55]:
train = pd.read_parquet(data_path + "/train.parquet")
valid = pd.read_parquet(data_path + "/valid.parquet")

In [ ]:
def undersample_background(df, min_signal_count=50):
    """
    Per dmr_ctype_label group:
    - Keep all signal reads (dmr_ctype_label == original_label)
    - Upsample signal to min_signal_count if below threshold
    - Undersample background (original_label != label) to match signal count
    """
    result = []

    for group_name, group_df in df.groupby('dmr_ctype_label'):
        signal = group_df[group_df['dmr_ctype_label'] == group_df['original_label']]
        background = group_df[group_df['original_label'] != group_df['label']]

        n_signal = len(signal)

        # Upsample small signal classes to minimum threshold
        if 0 < n_signal < min_signal_count:
            signal = signal.sample(n=min_signal_count, replace=True, random_state=42)
            n_signal = min_signal_count
        elif n_signal == 0:
            continue  # skip groups with no signal at all

        # Undersample background to match signal count (50/50)
        if len(background) > n_signal:
            background = background.sample(n=n_signal, random_state=42)

        result.append(signal)
        result.append(background)

    return pd.concat(result, ignore_index=True)

In [43]:
train_undersampled = undersample_background(train)
valid_undersampled = undersample_background(valid)

In [44]:
train_undersampled.to_parquet(data_path+"/train_undersampled_naive.parquet")
valid_undersampled.to_parquet(data_path+"/valid_undersampled_naive.parquet")

In [7]:
seq_length = 150
min_seq_length = 50

In [8]:
def prepare_methylbert_list(
    data_path, split, dmr_label_column, seq_length=150, min_seq_length=0, soft_labels = False
):
    data = pd.read_parquet(data_path + f"/{split}.parquet")
    # data = data.merge(dmrs, on=["dmr_label"])
    data_list = [['dna_seq', 'methyl_seq', 'dmr_ctype', 'dmr_label','ctype', 'original_label', 'on_target_mask']]
    for i,row in data.iterrows():
        dna = generate_kmer_str_with_overlap(row["seq"][:(seq_length+2)])
        methyl = row["pattern"][1:-1][:seq_length]
        if not len(dna) or not len(methyl):
            # print(row["input_ids"])
            # print(row["read_name"])
            continue
        label = row["soft_label"] if soft_labels else row["label"]
        o_label = row["original_label"]
        dmr_label = row[dmr_label_column]
        dmr_ctype = row["dmr_ctype_label"]
        data_list.append([dna, methyl, dmr_ctype,dmr_label,label,o_label,o_label==dmr_ctype])
    return data_list

In [45]:
# train, valid, test = [prepare_methylbert_list(data_path, split=x,dmrs=dmrs, seq_length=seq_length) for x in ["train", "valid", "test"]]
train, valid = [
    prepare_methylbert_list(
        data_path,
        split=x,
        dmr_label_column=dmr_label_column,
        seq_length=seq_length,
        min_seq_length=min_seq_length,
        soft_labels = False
    )
    for x in ["train_undersampled_naive", "valid_undersampled_naive"]
]
# train_dataset, valid_dataset, test_dataset = [MethylBertFinetuneDataset(data_source=x,vocab=MethylVocab(k=3),seq_len=seq_length, lazy_tokenization=True) for x in [train, valid, test]]
train_dataset, valid_dataset = [MethylBertFinetuneDataset(data_source=x,vocab=MethylVocab(k=3),seq_len=seq_length, lazy_tokenization=True, soft_labels=False) for x in [train, valid]]

Building Vocab
Total number of sequences: 158860
Using lazy tokenization (on-the-fly processing)
Building Vocab
Total number of sequences: 50388
Using lazy tokenization (on-the-fly processing)


### Step 1: Perform fine-tunning

In [46]:
import torch

In [47]:
torch.__version__

'2.7.1+cu128'

In [48]:
from collections import OrderedDict

In [49]:
rrms_config = OrderedDict([
    ("lr", 0.0004),
    ("beta", (0.9, 0.98)),
    ("weight_decay", 0.1),
    ("warmup_step", 100),
    ("eps", 1e-6),
    ("with_cuda", True),
    ("log_freq", 100),
    ("eval_freq", 100),
    ("n_hidden", None),
    ("decrease_steps", 200),
    ("eval", False),
    ("amp", True),
    ("gradient_accumulation_steps", 1),
    ("max_grad_norm", 1.0),
    ("save_freq", None),
    ("loss", "ce"),
    ("adam_beta1", 0.9),
    ("adam_beta2", 0.98),
    ("seed", 950410),
    ("focal_init", False),
    ("bg_class_index", 39),
    ("focal_prior_prob", 0.03)
])

In [50]:
model_instance = MethylBert(custom_config=rrms_config, 
                            foundation_model_path = "../foundationalModels/methylbert_hg19_12l",
                            load_weights = True,
                            num_labels=40,
                            num_dmr_labels = max(dmrs[dmr_label_column])+1,
                            seq_len=seq_length,
                            output_dir=f"methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_{dmr_label_column}_{data_prefix}_postfiltered_min_length_{min_seq_length}_hard_labels_from_soft_labels_split_undersampled_naive",
                            batch_size=512,
                            classifier_implementation="dmr_attention_based",
                            soft_labels = False)

Some weights of MethylBertEmbeddedDMR were not initialized from the model checkpoint at ../foundationalModels/methylbert_hg19_12l and are newly initialized: ['classifier.classifier.0.bias', 'classifier.classifier.0.weight', 'classifier.classifier.3.bias', 'classifier.classifier.3.weight', 'classifier.context_fusion.0.bias', 'classifier.context_fusion.0.weight', 'classifier.context_fusion.1.bias', 'classifier.context_fusion.1.weight', 'classifier.dmr_embedding.weight', 'classifier.key_proj.bias', 'classifier.key_proj.weight', 'classifier.query_proj.bias', 'classifier.query_proj.weight', 'classifier.value_proj.bias', 'classifier.value_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading MethylBertEmbeddedDMR with dmr_attention_based classifier from foundation path: ../foundationalModels/methylbert_hg19_12l
Cross Entropy loss assigned (multi-class)
Using attention-based classifier with DMR context


Setting biases for focal loss manually for now: 

In [ ]:
# import math
# prior_prob = 0.03
# num_fg = 39
# bg_bias = math.log((1.0 - prior_prob) / prior_prob * num_fg)
# model_instance.model.classifier.classifier[-1].bias = torch.nn.Parameter(torch.tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
#         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., bg_bias]))
# model_instance.model.classifier.classifier[-1].bias

In [51]:
# model_instance.fine_tune(data_path=data_path)
model_instance.fine_tune(
    data_path=None,
    train_dataset=train_dataset,
    val_dataset=valid_dataset,
    test_dataset=None,
)

/home/luna.kuleuven.be/u0169940/Repos/methyldl/methyldl/modelling/classifiers/methylbert.py:674: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


OUTPUT DIR IS: methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels_from_soft_labels_split_undersampled_naive


Step,Training Loss,Validation Loss,Accuracy,F1,Matthews Correlation,Precision,Recall
100,1.789000,0.263901,0.938180,0.876615,0.915856,0.903804,0.857355
200,0.234300,0.194343,0.941097,0.907953,0.919809,0.925334,0.893302
300,0.211500,0.191443,0.942586,0.926802,0.921810,0.950803,0.907243
400,0.206000,0.198185,0.935878,0.921208,0.913211,0.922102,0.922535
500,0.198400,0.185736,0.941712,0.926758,0.920691,0.942620,0.914383
600,0.196900,0.190974,0.940144,0.925574,0.918586,0.944395,0.910817
700,0.194200,0.182280,0.942367,0.928850,0.921475,0.956852,0.905598
800,0.190100,0.188711,0.940343,0.925699,0.918670,0.954678,0.902701
900,0.187500,0.194959,0.939172,0.926184,0.917677,0.928842,0.925997
1000,0.191800,0.183666,0.938021,0.921318,0.915760,0.937105,0.910165


KeyboardInterrupt: 

In [ ]:
model_instance = MethylBert(
    custom_config=rrms_config,
    foundation_model_path="hanyangii/methylbert_hg19_12l",
    num_labels=40,
    num_dmr_labels=max(dmrs["dmr_label"]) + 1,
    fine_tuned_model_path="methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_v2/checkpoint-1400/model.safetensors",
    classifier_implementation="dmr_attention_based",
)

In [ ]:
test = prepare_methylbert_list(
    data_path, split="test", dmrs=dmrs, seq_length=seq_length
)
test_dataset = MethylBertFinetuneDataset(
    data_source=test, vocab=MethylVocab(k=3), seq_len=seq_length, lazy_tokenization=True
)

In [ ]:
import pickle

In [ ]:
# with open("methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_v2/test_finetune_dataset.pkl","wb") as f:
#     pickle.dump(test_dataset, f)

with open(
    "methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_v2/test_finetune_dataset.pkl",
    "rb",
) as f:
    test_dataset = pickle.load(f)

In [ ]:
res = model_instance.predict(test_dataset, batch_size=2200)

In [ ]:
# with open("methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_v2/predictions_test.pkl", "wb") as f:
#     pickle.dump(res, f)

In [ ]:
with open(
    "methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_v2/predictions_test.pkl",
    "rb",
) as f:
    res = pickle.load(f)

In [ ]:
predictions = res[0].argmax(axis=1)
labels = res[1]
confidence = res[0].max(axis=1)

In [ ]:
predictions_filtered = predictions[labels != 39]
labels_filtered = labels[labels != 39]

In [ ]:
np.mean(predictions == labels)

In [ ]:
all_data = []
for split in ["train", "valid", "test"]:
    x = pd.read_parquet(data_path + f"/{split}.parquet")
    x["split"] = split
    all_data.append(x)

In [ ]:
all_data = pd.concat(all_data)

In [ ]:
labels_dict = {
    int(value["label"]): key
    for (key, value) in all_data[["label", "ctype"]]
    .groupby("ctype")
    .mean()
    .to_dict(orient="index")
    .items()
}

In [ ]:
split_metadata = (
    all_data[["label", "file", "split"]]
    .groupby(["label", "file", "split"])
    .count()
    .reset_index()
    .groupby(["label", "split"])
    .agg({"file": lambda x: list(x)})
    .reset_index()
)
split_metadata["n_files"] = split_metadata["file"].apply(len)
split_metadata["n_files"] = split_metadata["file"].apply(len)

In [ ]:
# Convert file lists to sets for faster intersection operations
split_metadata["file_set"] = split_metadata["file"].apply(set)

# Initialize the intersection columns
split_metadata["test_intersect"] = 0
split_metadata["train_intersect"] = 0
split_metadata["valid_intersect"] = 0

# Calculate intersections for each row
for idx, row in split_metadata.iterrows():
    current_label = row["label"]
    current_split = row["split"]
    current_files = row["file_set"]

    # For each split type
    for split_type in ["test", "train", "valid"]:
        # Skip if it's the same split as current row (self value = 0)
        if split_type == current_split:
            continue

        # Find all rows with same label and target split
        mask = (split_metadata["label"] == current_label) & (
            split_metadata["split"] == split_type
        )
        target_rows = split_metadata[mask]

        # Calculate total intersection count across all matching rows
        total_intersection = 0
        for _, target_row in target_rows.iterrows():
            intersection_count = len(current_files & target_row["file_set"])
            total_intersection += intersection_count

        # Set the appropriate column
        split_metadata.at[idx, f"{split_type}_intersect"] = total_intersection

# Drop the temporary file_set column
split_metadata = split_metadata.drop("file_set", axis=1)

In [ ]:
split_metadata["intersect_rate"] = [
    max(x[1][4], x[1][5], x[1][6]) for x in split_metadata.iterrows()
] / split_metadata["n_files"]
labels_dict = pd.DataFrame.from_dict(labels_dict, orient="index").reset_index()
labels_dict.columns = ["label", "Cell Type"]
split_metadata = (
    split_metadata.groupby("label").agg({"intersect_rate": np.mean}).reset_index()
)
labels_dict = pd.merge(labels_dict, split_metadata, on="label")
labels_dict["intersect_rate"] = np.round(labels_dict["intersect_rate"], 2)

In [ ]:
predictions_data = pd.DataFrame(
    zip(predictions, labels), columns=["predictions", "labels"]
)

In [ ]:
test_data = pd.read_parquet(data_path + f"/test.parquet")

In [ ]:
predictions_data.shape

In [ ]:
test[-1]

### Step 2: Test predictions

In [ ]:
from methyldl.data.dataset import generate_example_data_for_methylbert

In [ ]:
sequence_length = 150
synthetic_data = generate_example_data_for_methylbert(
    sequence_length=sequence_length,
    include_cpg_methylation=True,
    include_m6a_methylation=False,
    include_labels=True,
    num_samples=2,  # Single sample per repeat
)

In [ ]:
synthetic_data

In [ ]:
test_data = MethylBertFinetuneDataset(
    data_source=synthetic_data, vocab=MethylVocab(k=3), seq_len=150
)

In [ ]:
# test_data = MethylBertFinetuneDataset(data_source=data_path+"/test.txt",
#                                       vocab=MethylVocab(k=3),
#                             seq_len=150)

In [ ]:
res = model_instance.predict(test_data)

In [ ]:
from sklearn.metrics import confusion_matrix, auc, roc_curve
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Set Matplotlib styling
plt.rcParams.update(
    {
        "font.size": 12,
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "grid.color": "lightgray",
        "grid.linestyle": "-",
    }
)

# Compute confusion matrix
cf_matrix = confusion_matrix(res.label_ids, res.predictions > 0.5)

# Create the heatmap using Matplotlib
fig, ax = plt.subplots(1, figsize=(6, 6))
cax = ax.matshow(cf_matrix, cmap="PiYG")
plt.colorbar(cax)

# Annotate the heatmap
for (i, j), val in np.ndenumerate(cf_matrix):
    ax.text(
        j,
        i,
        f"{val}",
        ha="center",
        va="center",
        color="white" if val > cf_matrix.max() / 2 else "black",
    )

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=12)
ax.set_ylabel("Ground-truth", fontsize=12)
ax.set_xticks([0, 1])
ax.set_xticklabels(["N", "T"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["N", "T"])

# Compute ROC and AUC
fpr, tpr, thresholds = roc_curve(res.label_ids, res.predictions)
auc_val = auc(fpr, tpr)

# Set title with AUC value
ax.set_title(f"MethylBERT test (auc: {auc_val:.3f})", fontsize=14)

# Show plot
plt.tight_layout()
plt.show()

In [ ]:
methylbert_formated_data = MethylBertFinetuneDataset(
    data_source=synthetic_data, vocab=MethylVocab(k=3), seq_len=150
)

In [ ]:
model_instance.predict(methylbert_formated_data)

### Loyfer Analysis

In [ ]:
import pickle

In [ ]:
with open(
    "methylBertLoyferWithReject_vanillaClassifier_DMRs_stratified/predictions_test.pkl",
    "rb",
) as f:
    res_vanilla = pickle.load(f)

In [ ]:
with open(
    "methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_v2/predictions_test.pkl",
    "rb",
) as f:
    res_attention = pickle.load(f)

In [ ]:
predictions = np.argmax(res_attention[0], 1)
labels = res_attention[1]
confidence = res_attention[0].max(axis=1)

In [ ]:
data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/TrainingDataWithRejectionDMRsStratified/"
all_data = []
for split in ["train", "valid", "test"]:
    x = pd.read_parquet(data_path + f"/{split}.parquet")
    x["split"] = split
    all_data.append(x)

In [ ]:
all_data = pd.concat(all_data)

In [ ]:
from copy import deepcopy

In [ ]:
test_data = deepcopy(all_data[all_data["split"] == "test"])

In [ ]:
test_data["methylated_CpGs"] = test_data["original_methyl"].apply(
    lambda x: x.count("C")
)
test_data["unmethylated_CpGs"] = test_data["original_methyl"].apply(
    lambda x: x.count("T")
)
test_data["methylation_level"] = test_data["methylated_CpGs"] / (
    test_data["methylated_CpGs"] + test_data["unmethylated_CpGs"]
)

In [ ]:
test_data["prediction"] = predictions

In [ ]:
test_data["confidence"] = confidence

In [ ]:
labels_dict = {
    int(value["original_label"]): key
    for (key, value) in all_data[["original_label", "ctype"]]
    .groupby("ctype")
    .mean()
    .to_dict(orient="index")
    .items()
}

In [ ]:
labels_list = [labels_dict[i] for i in range(39)]

In [ ]:
labels_list.append("Rejected")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [ ]:
# Set Matplotlib styling
target_confidence = 0.5
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "grid.color": "lightgray",
        "grid.linestyle": "-",
    }
)

confident_predictions = np.array(
    [int(x) if y > target_confidence else 39 for (x, y) in zip(predictions, confidence)]
)
# 1. Compute confusion matrix
cf_matrix = confusion_matrix(labels, confident_predictions)

# 2. Logic to exclude last row/col from Color Scaling
# We slice the matrix to get everything EXCEPT the last row and last column
subset_matrix = cf_matrix[:-1, :-1]
# Find the max value in the specific cell types
max_val_subset = np.max(subset_matrix)

# Create the heatmap
fig, ax = plt.subplots(1, figsize=(20, 20))

# 3. Apply the scaling using vmax
# 'vmax' clamps the color range. Anything higher than this (i.e., the last row/col)
# will appear as the darkest color (saturated), but won't distort the gradient for the rest.
cax = ax.matshow(cf_matrix, cmap="PuRd", vmin=0, vmax=max_val_subset)

# Add colorbar
# extend='max' indicates that values exist beyond the top of the colorbar
plt.colorbar(cax, fraction=0.046, pad=0.04, extend="max")

# 4. Annotate the heatmap
# We adjust the text color threshold.
# Since the last row/col is saturated, we force white text there for readability.
for (i, j), val in np.ndenumerate(cf_matrix):
    # Determine text color:
    # If we are in the "Background" zone (last row or col), usually dark color -> use White text
    # Else, use standard thresholding based on the subset max
    if i == cf_matrix.shape[0] - 1 or j == cf_matrix.shape[1] - 1:
        color = (
            "white" if val > (max_val_subset * 0.3) else "black"
        )  # Adjust contrast as needed
    else:
        color = "white" if val > (max_val_subset / 2) else "black"

    # Optional: If numbers are huge, use scientific notation or hide zeros
    label_text = f"{val}" if val > 0 else ""
    ax.text(j, i, label_text, ha="center", va="center", color=color, fontsize=8)

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=16, labelpad=20)
ax.set_ylabel("Ground-truth", fontsize=16, labelpad=20)

ax.set_xticks(range(len(labels_list)))
ax.set_xticklabels(labels_list, rotation=90, fontsize=8)  # Rotated for readability
ax.set_yticks(range(len(labels_list)))
ax.set_yticklabels(labels_list, fontsize=8)

# Move x-axis ticks to bottom (matshow puts them on top by default)
ax.xaxis.set_ticks_position("bottom")

# Set title
ax.set_title(
    f"MethylBERT: Attention Classifier (Color scaled to {max_val_subset})",
    fontsize=14,
    pad=20,
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 1 - remove matched rejected labels from the CM
# Step 2 - for each remaining rejected GT, restore original label

In [ ]:
mask = (labels == 39) & (confident_predictions == labels)
mask = np.array([not x for x in mask])

In [ ]:
test_size = len(test_dataset.raw_lines)

In [ ]:
original_labels = np.array(
    [int(test_dataset.raw_lines[i].split("\t")[5]) for i in range(test_size)]
)
reference_labels = np.array(
    [int(test_dataset.raw_lines[i].split("\t")[4]) for i in range(test_size)]
)

In [ ]:
predictions_filtered = confident_predictions[mask]
labels_filtered = labels[mask]
original_labels_filtered = original_labels[mask]

In [ ]:
# Set Matplotlib styling
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "grid.color": "lightgray",
        "grid.linestyle": "-",
    }
)

# 1. Compute confusion matrix
cf_matrix = confusion_matrix(original_labels_filtered, predictions_filtered)

# 2. Logic to exclude last row/col from Color Scaling
# We slice the matrix to get everything EXCEPT the last row and last column
subset_matrix = cf_matrix[:-1, :-1]
# Find the max value in the specific cell types
max_val_subset = np.max(subset_matrix)

# Create the heatmap
fig, ax = plt.subplots(1, figsize=(20, 20))

# 3. Apply the scaling using vmax
# 'vmax' clamps the color range. Anything higher than this (i.e., the last row/col)
# will appear as the darkest color (saturated), but won't distort the gradient for the rest.
cax = ax.matshow(cf_matrix, cmap="PuRd", vmin=0, vmax=max_val_subset)

# Add colorbar
# extend='max' indicates that values exist beyond the top of the colorbar
plt.colorbar(cax, fraction=0.046, pad=0.04, extend="max")

# 4. Annotate the heatmap
# We adjust the text color threshold.
# Since the last row/col is saturated, we force white text there for readability.
for (i, j), val in np.ndenumerate(cf_matrix):
    # Determine text color:
    # If we are in the "Background" zone (last row or col), usually dark color -> use White text
    # Else, use standard thresholding based on the subset max
    if i == cf_matrix.shape[0] - 1 or j == cf_matrix.shape[1] - 1:
        color = (
            "white" if val > (max_val_subset * 0.3) else "black"
        )  # Adjust contrast as needed
    else:
        color = "white" if val > (max_val_subset / 2) else "black"

    # Optional: If numbers are huge, use scientific notation or hide zeros
    label_text = f"{val}" if val > 0 else ""
    ax.text(j, i, label_text, ha="center", va="center", color=color, fontsize=8)

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=16, labelpad=20)
ax.set_ylabel("Ground-truth", fontsize=16, labelpad=20)

ax.set_xticks(range(len(labels_list)))
ax.set_xticklabels(labels_list, rotation=90, fontsize=8)  # Rotated for readability
ax.set_yticks(range(len(labels_list)))
ax.set_yticklabels(labels_list, fontsize=8)

# Move x-axis ticks to bottom (matshow puts them on top by default)
ax.xaxis.set_ticks_position("bottom")

# Set title
ax.set_title(
    f"MethylBERT: Attention Classifier (Color scaled to {max_val_subset})",
    fontsize=14,
    pad=20,
)

plt.tight_layout()
plt.show()

##### Box-plots of different predicted groups methylation levels
1. Correctly predicted
2. Incorrectly rejected
3. Incorrectly predicted 
4. Correctly rejected

In [ ]:
target_confidence = 0.565
test_data["prediction"] = np.array(
    [int(x) if y > target_confidence else 39 for (x, y) in zip(predictions, confidence)]
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def create_methylation_boxplots(df, min_methyl_labels_count=0):
    """
    Create four box plots of methylation levels based on prediction outcomes.

    Parameters:
    df: DataFrame with 'label', 'prediction', and 'methylation_level' columns
    """
    df = df[df["total_methylated_labels"] > min_methyl_labels_count]
    # Define the four groups
    correctly_predicted = df[(df["label"] == df["prediction"]) & (df["label"] != 39)]
    incorrectly_rejected = df[(df["label"] != 39) & (df["prediction"] == 39)]
    incorrectly_predicted = df[(df["label"] == 39) & (df["prediction"] != 39)]
    correctly_rejected = df[(df["label"] == 39) & (df["prediction"] == 39)]

    # Print sample sizes
    print("Sample sizes:")
    print(
        f"1. Correctly predicted (label == prediction, label != 39): {len(correctly_predicted)}"
    )
    print(
        f"2. Incorrectly rejected (label != 39, prediction == 39): {len(incorrectly_rejected)}"
    )
    print(
        f"3. Incorrectly predicted (label == 39, prediction != 39): {len(incorrectly_predicted)}"
    )
    print(
        f"4. Correctly rejected (label == prediction == 39): {len(correctly_rejected)}"
    )
    print()

    # Prepare data for plotting
    plot_data = []
    groups = []

    for data, name in [
        (correctly_predicted, "Correctly\nPredicted"),
        (incorrectly_rejected, "Incorrectly\nRejected"),
        (incorrectly_predicted, "Incorrectly\nPredicted"),
        (correctly_rejected, "Correctly\nRejected"),
    ]:
        if len(data) > 0:
            plot_data.append(data["methylation_level"].values)
            groups.append(name)
        else:
            plot_data.append([])
            groups.append(name)

    # Create the box plot
    fig, ax = plt.subplots(figsize=(12, 7))

    # Create box plot
    bp = ax.boxplot(
        plot_data, labels=groups, patch_artist=True, showmeans=True, meanline=True
    )

    # Customize colors
    colors = ["#2ecc71", "#e74c3c", "#e67e22", "#3498db"]
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    # Customize plot
    ax.set_ylabel("Methylation Level", fontsize=12, fontweight="bold")
    ax.set_xlabel("Prediction Groups", fontsize=12, fontweight="bold")
    ax.set_title(
        "Methylation Level Distribution by Prediction Outcome\n(label 39 = rejection class)",
        fontsize=14,
        fontweight="bold",
        pad=20,
    )
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    # Add sample size annotations
    for i, (data, group) in enumerate(zip(plot_data, groups), 1):
        ax.text(
            i,
            ax.get_ylim()[0],
            f"n={len(data)}",
            ha="center",
            va="top",
            fontsize=9,
            style="italic",
        )

    plt.tight_layout()

    # Print summary statistics
    print("\nSummary Statistics:")
    print("-" * 80)
    for data, name in zip(plot_data, groups):
        if len(data) > 0:
            print(f"\n{name.replace(chr(10), ' ')}:")
            print(f"  Mean: {np.mean(data):.4f}")
            print(f"  Median: {np.median(data):.4f}")
            print(f"  Std: {np.std(data):.4f}")
            print(f"  Min: {np.min(data):.4f}")
            print(f"  Max: {np.max(data):.4f}")
        else:
            print(f"\n{name.replace(chr(10), ' ')}: No data")
    ax.tick_params(axis="x", pad=20)
    plt.show()

    return fig, ax


def create_methylation_violinplots(
    df,
    title="Methylation Level Distribution by Prediction Outcome\n(label 39 = rejection class)",
):
    """
    Create four violin plots of methylation levels based on prediction outcomes.

    Parameters:
    df: DataFrame with 'label', 'prediction', and 'methylation_level' columns
    """

    # Define the four groups
    correctly_predicted = df[(df["label"] == df["prediction"]) & (df["label"] != 39)]
    incorrectly_rejected = df[(df["label"] != 39) & (df["prediction"] == 39)]
    incorrectly_predicted = df[
        (df["label"] == 39)
        & (df["prediction"] != 39)
        & (df["prediction"] != df["original_label"])
    ]
    correctly_rejected = df[(df["label"] == 39) & (df["prediction"] == 39)]

    # Print sample sizes
    print("Sample sizes:")
    print(
        f"1. Correctly predicted (label == prediction, label != 39): {len(correctly_predicted)}"
    )
    print(
        f"2. Incorrectly rejected (label != 39, prediction == 39): {len(incorrectly_rejected)}"
    )
    print(
        f"3. Incorrectly predicted (label == 39, prediction != 39): {len(incorrectly_predicted)}"
    )
    print(
        f"4. Correctly rejected (label == prediction == 39): {len(correctly_rejected)}"
    )
    print()

    # Prepare data for plotting
    plot_data = []

    for data, group_num, name in [
        (correctly_predicted, 1, "Correctly\nPredicted"),
        (incorrectly_rejected, 2, "Incorrectly\nRejected"),
        (incorrectly_predicted, 3, "Incorrectly\nPredicted"),
        (correctly_rejected, 4, "Correctly\nRejected"),
    ]:
        if len(data) > 0:
            temp_df = pd.DataFrame(
                {
                    "methylation_level": data["methylation_level"].values,
                    "group": name,
                    "group_num": group_num,
                }
            )
            plot_data.append(temp_df)

    # Combine all data
    plot_df = pd.concat(plot_data, ignore_index=True)

    # Create the violin plot
    fig, ax = plt.subplots(figsize=(14, 8))

    # Define colors
    colors = ["#2ecc71", "#e74c3c", "#e67e22", "#3498db"]

    # Create violin plot
    parts = ax.violinplot(
        [
            plot_df[plot_df["group_num"] == i]["methylation_level"].values
            for i in range(1, 5)
        ],
        positions=range(1, 5),
        showmeans=True,
        showmedians=True,
        widths=0.7,
    )

    # Color the violins
    for i, (pc, color) in enumerate(zip(parts["bodies"], colors)):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
        pc.set_edgecolor("black")
        pc.set_linewidth(1.5)

    # Customize other elements
    for partname in ("cbars", "cmins", "cmaxes", "cmedians", "cmeans"):
        if partname in parts:
            vp = parts[partname]
            vp.set_edgecolor("black")
            vp.set_linewidth(1.5)

    # Set x-tick labels with group names
    group_labels = [
        "Correctly\nPredicted",
        "Incorrectly\nRejected",
        "Incorrectly\nPredicted",
        "Correctly\nRejected",
    ]
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels(group_labels, fontsize=11, fontweight="bold")

    # Add sample sizes below group labels
    sample_sizes = [
        len(correctly_predicted),
        len(incorrectly_rejected),
        len(incorrectly_predicted),
        len(correctly_rejected),
    ]

    # Increase padding between tick labels and axis
    ax.tick_params(axis="x", pad=15, labelsize=11)
    ax.tick_params(axis="y", pad=8, labelsize=10)

    # Add sample size annotations with extra spacing
    y_min = ax.get_ylim()[0]
    y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
    annotation_y = y_min - (0.08 * y_range)  # Position below x-axis

    for i, n in enumerate(sample_sizes, 1):
        ax.text(
            i,
            annotation_y,
            f"(n={n})",
            ha="center",
            va="top",
            fontsize=10,
            style="italic",
            fontweight="bold",
        )

    # Customize plot
    ax.set_ylabel("Methylation Level", fontsize=13, fontweight="bold", labelpad=12)
    ax.set_xlabel("Prediction Groups", fontsize=13, fontweight="bold", labelpad=25)
    ax.set_title(title, fontsize=15, fontweight="bold", pad=20)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.8)

    # Adjust y-axis to make room for sample size labels
    y_lim = ax.get_ylim()
    ax.set_ylim([y_lim[0] - (0.12 * y_range), y_lim[1]])

    # Add legend
    from matplotlib.patches import Patch

    legend_elements = [
        Patch(
            facecolor=colors[0],
            alpha=0.7,
            edgecolor="black",
            label="Correctly Predicted",
        ),
        Patch(
            facecolor=colors[1],
            alpha=0.7,
            edgecolor="black",
            label="Incorrectly Rejected",
        ),
        Patch(
            facecolor=colors[2],
            alpha=0.7,
            edgecolor="black",
            label="Incorrectly Predicted",
        ),
        Patch(
            facecolor=colors[3],
            alpha=0.7,
            edgecolor="black",
            label="Correctly Rejected",
        ),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper left",
        bbox_to_anchor=(1, 1),
        fontsize=10,
        framealpha=0.9,
    )

    plt.tight_layout()

    # Print summary statistics
    print("\nSummary Statistics:")
    print("-" * 80)
    for data, name in [
        (correctly_predicted, "Correctly Predicted"),
        (incorrectly_rejected, "Incorrectly Rejected"),
        (incorrectly_predicted, "Incorrectly Predicted"),
        (correctly_rejected, "Correctly Rejected"),
    ]:
        if len(data) > 0:
            print(f"\n{name}:")
            print(f"  Count: {len(data)}")
            print(f"  Mean: {data['methylation_level'].mean():.4f}")
            print(f"  Median: {data['methylation_level'].median():.4f}")
            print(f"  Std: {data['methylation_level'].std():.4f}")
            print(f"  Min: {data['methylation_level'].min():.4f}")
            print(f"  Max: {data['methylation_level'].max():.4f}")
            print(f"  Q1 (25%): {data['methylation_level'].quantile(0.25):.4f}")
            print(f"  Q3 (75%): {data['methylation_level'].quantile(0.75):.4f}")
        else:
            print(f"\n{name}: No data")

    plt.show()

    return fig, ax

In [ ]:
test_data["total_methylated_labels"] = (
    test_data["methylated_CpGs"] + test_data["unmethylated_CpGs"]
)

In [ ]:
create_methylation_violinplots(test_data)

##### Box-plot for naive classifier that is tresholding based on methylation status
Assignes label equal to dmr_label whenever read is hypomethylated; otherwise rejects:

In [ ]:
test_data["prediction"] = np.array(
    [int(x) if y > target_confidence else 39 for (x, y) in zip(predictions, confidence)]
)

In [ ]:
test_data["prediction"] = np.array(
    [
        x if y < 0.1 else 39
        for (x, y) in zip(test_data["dmr_ctype_label"], test_data["methylation_level"])
    ]
)

In [ ]:
create_methylation_violinplots(
    test_data,
    title="Naive classifier that sets label equal to DMR label for all reads with B-value<0.1, otherwise rejects",
)

### Aggregating predictions

In [ ]:
res_attention_pd = pd.DataFrame(
    res_attention[0], columns=["prediction_" + str(x) for x in range(40)]
)

In [ ]:
test_data = pd.concat([test_data, res_attention_pd], axis=1)

In [ ]:
selected_columns = ["prediction_" + str(x) for x in range(40)]
selected_columns.extend(["dmr_label", "dmr_ctype", "file", "label", "orginal_label"])

In [ ]:
test_data["total_marked_cpgs"] = (
    test_data["methylated_CpGs"] + test_data["unmethylated_CpGs"]
)

In [ ]:
import pandas as pd
import numpy as np
from typing import List, Optional, Union


def aggregate_predictions_by_dmr(
    df: pd.DataFrame,
    group_cols: List[str] = ["dmr_label", "file", "original_label"],
    prediction_cols: Optional[List[str]] = None,
    weight_col: str = "total_marked_cpgs",
    create_weight_from_cpgs: bool = True,
) -> pd.DataFrame:
    """
    Aggregate predictions for each class across all reads at the DMR level.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with read-level predictions.
    group_cols : List[str]
        Columns to group by for aggregation. Default: ['dmr_label', 'file', 'original_label']
    prediction_cols : Optional[List[str]]
        List of prediction column names. If None, auto-detects columns starting with 'prediction_'.
    weight_col : str
        Column name to use for weighted aggregation. Default: 'total_marked_cpgs'
    create_weight_from_cpgs : bool
        If True and weight_col doesn't exist, creates it from methylated_CpGs + unmethylated_CpGs.

    Returns
    -------
    pd.DataFrame
        Aggregated dataframe with both simple average and weighted average predictions.
    """

    df = df.copy()

    # Auto-detect prediction columns if not provided
    if prediction_cols is None:
        prediction_cols = [
            col
            for col in df.columns
            if col.startswith("prediction_") and col != "prediction"
        ]
        prediction_cols = sorted(prediction_cols, key=lambda x: int(x.split("_")[1]))
        prediction_cols.append("methylation_level")

    # Create weight column if needed
    if weight_col not in df.columns and create_weight_from_cpgs:
        if "methylated_CpGs" in df.columns and "unmethylated_CpGs" in df.columns:
            df[weight_col] = df["methylated_CpGs"] + df["unmethylated_CpGs"]
        else:
            raise ValueError(
                f"Weight column '{weight_col}' not found and cannot create from CpG columns."
            )

    # Handle case where weights might be 0 (avoid division by zero)
    df["_weight"] = df[weight_col].clip(lower=1e-10)

    # Simple average aggregation
    simple_avg = df.groupby(group_cols)[prediction_cols].mean()
    simple_avg.columns = [f"{col}_avg" for col in simple_avg.columns]

    # Weighted average aggregation
    def weighted_average(group):
        weights = group["_weight"].values
        total_weight = weights.sum()

        result = {}
        for col in prediction_cols:
            values = group[col].values
            result[f"{col}_wavg"] = np.average(values, weights=weights)

        result["total_weight"] = total_weight
        result["n_reads"] = len(group)
        return pd.Series(result)

    weighted_avg = df.groupby(group_cols).apply(weighted_average, include_groups=False)

    # Combine results
    result = simple_avg.join(weighted_avg)

    # Add additional metadata columns (take first value per group)
    metadata_cols = [
        col
        for col in df.columns
        if col not in prediction_cols + group_cols + ["_weight", weight_col]
        and col in ["ctype", "dmr_ctype", "label", "chromosome"]
    ]

    if metadata_cols:
        metadata = df.groupby(group_cols)[metadata_cols].first()
        result = result.join(metadata)

    return result.reset_index()


def get_final_prediction(
    aggregated_df: pd.DataFrame,
    methods: list = ["avg", "wavg"],
    prediction_prefix: str = "prediction_",
) -> pd.DataFrame:
    """
    Get final class prediction from aggregated probabilities.

    Parameters
    ----------
    aggregated_df : pd.DataFrame
        Output from aggregate_predictions_by_dmr()
    method : str
        'avg' for simple average or 'wavg' for weighted average
    prediction_prefix : str
        Prefix for prediction columns

    Returns
    -------
    pd.DataFrame
        DataFrame with added final_prediction and final_confidence columns.
    """

    df = aggregated_df.copy()
    for method in methods:
        suffix = f"_{method}"

        # Find relevant prediction columns
        pred_cols = [
            col
            for col in df.columns
            if col.startswith(prediction_prefix) and col.endswith(suffix)
        ]
        pred_cols = sorted(
            pred_cols,
            key=lambda x: int(x.replace(prediction_prefix, "").replace(suffix, "")),
        )

        if not pred_cols:
            raise ValueError(f"No prediction columns found with suffix '{suffix}'")

        # Get predictions as array
        pred_array = df[pred_cols].values

        # Final prediction is argmax, confidence is max probability
        df[f"final_prediction_{method}"] = pred_array.argmax(axis=1)
        df[f"final_confidence_{method}"] = pred_array.max(axis=1)

    return df

#### Individual DMRs

In [ ]:
aggregated_predictions = aggregate_predictions_by_dmr(test_data)

In [ ]:
final_predictions = get_final_prediction(aggregated_predictions)

In [ ]:
# final_predictions = final_predictions[['dmr_label', 'file', 'original_label', 'total_weight', 'n_reads', 'chromosome',
#        'dmr_ctype', 'ctype', 'label', 'final_prediction_avg',
#        'final_confidence_avg', 'final_prediction_wavg',
#        'final_confidence_wavg', 'methylation_level_avg', 'methylation_level_wavg']]

In [ ]:
final_predictions_avg = final_predictions[
    [
        "label",
        "original_label",
        "final_prediction_avg",
        "final_confidence_avg",
        "methylation_level_avg",
    ]
]
final_predictions_avg.columns = [
    "label",
    "original_label",
    "prediction",
    "confidence",
    "methylation_level",
]

final_predictions_wavg = final_predictions[
    [
        "label",
        "original_label",
        "final_prediction_wavg",
        "final_confidence_wavg",
        "methylation_level_wavg",
    ]
]
final_predictions_wavg.columns = [
    "label",
    "original_label",
    "prediction",
    "confidence",
    "methylation_level",
]

In [ ]:
create_methylation_violinplots(
    final_predictions_avg, title="Averaging of predictions by DMR and file of origin"
)

In [ ]:
from copy import deepcopy

In [ ]:
create_methylation_violinplots(
    final_predictions_wavg,
    title="CpG Weighted averaging of predictions by DMR and file of origin",
)

In [ ]:
final_predictions_wavg_confident = deepcopy(final_predictions_wavg)
target_confidence = 0.8
final_predictions_wavg_confident["prediction"] = np.array(
    [
        int(x) if y > target_confidence else 39
        for (x, y) in zip(
            final_predictions_wavg["prediction"], final_predictions_wavg["confidence"]
        )
    ]
)

In [ ]:
create_methylation_violinplots(
    final_predictions_wavg_confident,
    title=f"CpG Weighted averaging of predictions by DMR and file of origin with minimum confidence: {target_confidence}",
)

#### Cell-specific DMRs versus all other DMRs

In [ ]:
test_data["is_cell_informative_region"] = (
    test_data["original_label"] == test_data["dmr_ctype_label"]
)

In [ ]:
aggregated_predictions_by_file = aggregate_predictions_by_dmr(
    test_data, group_cols=["file", "original_label", "is_cell_informative_region"]
)

In [ ]:
final_predictions_by_file = get_final_prediction(aggregated_predictions_by_file)

In [ ]:
final_predictions_by_file = final_predictions_by_file[
    [
        "label",
        "original_label",
        "final_prediction_wavg",
        "final_confidence_wavg",
        "methylation_level_wavg",
    ]
]
final_predictions_by_file.columns = [
    "label",
    "original_label",
    "prediction",
    "confidence",
    "methylation_level",
]

In [ ]:
# Set Matplotlib styling
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "grid.color": "lightgray",
        "grid.linestyle": "-",
    }
)

labels, confident_predictions = (
    final_predictions_by_file["label"],
    final_predictions_by_file["prediction"],
)
# 1. Compute confusion matrix
cf_matrix = confusion_matrix(labels, confident_predictions)

# 2. Logic to exclude last row/col from Color Scaling
# We slice the matrix to get everything EXCEPT the last row and last column
subset_matrix = cf_matrix[:-1, :-1]
# Find the max value in the specific cell types
max_val_subset = np.max(subset_matrix)

# Create the heatmap
fig, ax = plt.subplots(1, figsize=(10, 10))

# 3. Apply the scaling using vmax
# 'vmax' clamps the color range. Anything higher than this (i.e., the last row/col)
# will appear as the darkest color (saturated), but won't distort the gradient for the rest.
cax = ax.matshow(cf_matrix, cmap="PuRd", vmin=0, vmax=max_val_subset)

# Add colorbar
# extend='max' indicates that values exist beyond the top of the colorbar
plt.colorbar(cax, fraction=0.046, pad=0.04, extend="max")

# 4. Annotate the heatmap
# We adjust the text color threshold.
# Since the last row/col is saturated, we force white text there for readability.
for (i, j), val in np.ndenumerate(cf_matrix):
    # Determine text color:
    # If we are in the "Background" zone (last row or col), usually dark color -> use White text
    # Else, use standard thresholding based on the subset max
    if i == cf_matrix.shape[0] - 1 or j == cf_matrix.shape[1] - 1:
        color = (
            "white" if val > (max_val_subset * 0.3) else "black"
        )  # Adjust contrast as needed
    else:
        color = "white" if val > (max_val_subset / 2) else "black"

    # Optional: If numbers are huge, use scientific notation or hide zeros
    label_text = f"{val}" if val > 0 else ""
    ax.text(j, i, label_text, ha="center", va="center", color=color, fontsize=8)

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=16, labelpad=20)
ax.set_ylabel("Ground-truth", fontsize=16, labelpad=20)

ax.set_xticks(range(len(labels_list)))
ax.set_xticklabels(labels_list, rotation=90, fontsize=8)  # Rotated for readability
ax.set_yticks(range(len(labels_list)))
ax.set_yticklabels(labels_list, fontsize=8)

# Move x-axis ticks to bottom (matshow puts them on top by default)
ax.xaxis.set_ticks_position("bottom")

# Set title
ax.set_title(
    f"MethylBERT: Attention Classifier: Regions aggregated resuts", fontsize=14, pad=20
)

plt.tight_layout()
plt.show()

#### Cell-specific DMR regions aggregation

In [ ]:
aggregated_predictions_by_groupped_dmrs = aggregate_predictions_by_dmr(
    test_data,
    group_cols=[
        "file",
        "original_label",
        "is_cell_informative_region",
        "dmr_ctype_label",
    ],
)

In [ ]:
final_predictions_by_dmr_groups = get_final_prediction(
    aggregated_predictions_by_groupped_dmrs
)

In [ ]:
final_predictions_by_dmr_groups = final_predictions_by_dmr_groups[
    [
        "label",
        "original_label",
        "final_prediction_wavg",
        "final_confidence_wavg",
        "methylation_level_wavg",
        "dmr_ctype_label",
        "is_cell_informative_region",
        "file",
        "n_reads",
        "total_weight",
    ]
]
final_predictions_by_dmr_groups.columns = [
    "label",
    "original_label",
    "prediction",
    "confidence",
    "methylation_level",
    "dmr_ctype_label",
    "is_cell_informative_region",
    "file",
    "n_reads",
    "total_weight",
]

In [ ]:
# Set Matplotlib styling
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "grid.color": "lightgray",
        "grid.linestyle": "-",
    }
)

labels, confident_predictions = (
    final_predictions_by_dmr_groups["label"],
    final_predictions_by_dmr_groups["prediction"],
)
# 1. Compute confusion matrix
cf_matrix = confusion_matrix(labels, confident_predictions)

# 2. Logic to exclude last row/col from Color Scaling
# We slice the matrix to get everything EXCEPT the last row and last column
subset_matrix = cf_matrix[:-1, :-1]
# Find the max value in the specific cell types
max_val_subset = np.max(subset_matrix)

# Create the heatmap
fig, ax = plt.subplots(1, figsize=(10, 10))

# 3. Apply the scaling using vmax
# 'vmax' clamps the color range. Anything higher than this (i.e., the last row/col)
# will appear as the darkest color (saturated), but won't distort the gradient for the rest.
cax = ax.matshow(cf_matrix, cmap="PuRd", vmin=0, vmax=max_val_subset)

# Add colorbar
# extend='max' indicates that values exist beyond the top of the colorbar
plt.colorbar(cax, fraction=0.046, pad=0.04, extend="max")

# 4. Annotate the heatmap
# We adjust the text color threshold.
# Since the last row/col is saturated, we force white text there for readability.
for (i, j), val in np.ndenumerate(cf_matrix):
    # Determine text color:
    # If we are in the "Background" zone (last row or col), usually dark color -> use White text
    # Else, use standard thresholding based on the subset max
    if i == cf_matrix.shape[0] - 1 or j == cf_matrix.shape[1] - 1:
        color = (
            "white" if val > (max_val_subset * 0.3) else "black"
        )  # Adjust contrast as needed
    else:
        color = "white" if val > (max_val_subset / 2) else "black"

    # Optional: If numbers are huge, use scientific notation or hide zeros
    label_text = f"{val}" if val > 0 else ""
    ax.text(j, i, label_text, ha="center", va="center", color=color, fontsize=8)

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=16, labelpad=20)
ax.set_ylabel("Ground-truth", fontsize=16, labelpad=20)

ax.set_xticks(range(len(labels_list)))
ax.set_xticklabels(labels_list, rotation=90, fontsize=8)  # Rotated for readability
ax.set_yticks(range(len(labels_list)))
ax.set_yticklabels(labels_list, fontsize=8)

# Move x-axis ticks to bottom (matshow puts them on top by default)
ax.xaxis.set_ticks_position("bottom")

# Set title
ax.set_title(
    f"MethylBERT: Attention Classifier: Regions aggregated resuts", fontsize=14, pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
final_predictions_by_dmr_groups[
    final_predictions_by_dmr_groups["file"]
    == "GSM5652176_Adipocytes-Z000000T7_reads.csv"
]

In [ ]:
final_predictions_by_dmr_groups_agg = final_predictions_by_dmr_groups.groupby(
    ["file", "original_label", "label"]
).aggregate(
    {
        "prediction": lambda x: list(x),
        "confidence": lambda x: list(x),
        "dmr_ctype_label": lambda x: list(x),
    }
)

In [ ]:
final_predictions_by_dmr_groups_agg.reset_index(inplace=True)

In [ ]:
mask = final_predictions_by_dmr_groups_agg["prediction"].apply(
    lambda x: [element != 39 for element in x]
)

In [ ]:
final_predictions_by_dmr_groups_agg["predictions_without_rejection"] = [
    np.array(x)[np.array(y)]
    for (x, y) in zip(final_predictions_by_dmr_groups_agg["prediction"], mask)
]
final_predictions_by_dmr_groups_agg["confidences_without_rejection"] = [
    np.array(x)[np.array(y)]
    for (x, y) in zip(final_predictions_by_dmr_groups_agg["confidence"], mask)
]
final_predictions_by_dmr_groups_agg["dmr_ctype_labels_without_rejection"] = [
    np.array(x)[np.array(y)]
    for (x, y) in zip(final_predictions_by_dmr_groups_agg["dmr_ctype_label"], mask)
]

In [ ]:
final_predictions_by_dmr_groups_agg

In [ ]:
final_predictions_by_dmr_groups_agg["final_prediction"] = [
    np.array(x)[np.argmax(np.array(y))] if len(x) else 39
    for (x, y) in zip(
        final_predictions_by_dmr_groups_agg["predictions_without_rejection"],
        final_predictions_by_dmr_groups_agg["confidences_without_rejection"],
    )
]

In [ ]:
final_predictions_by_dmr_groups_agg["final_confidence"] = [
    np.array(x)[np.argmax(np.array(y))] if len(x) else 1
    for (x, y) in zip(
        final_predictions_by_dmr_groups_agg["confidences_without_rejection"],
        final_predictions_by_dmr_groups_agg["confidences_without_rejection"],
    )
]

In [ ]:
final_predictions_by_dmr_groups_agg

In [ ]:
final_predictions_by_dmr_groups_agg = final_predictions_by_dmr_groups_agg[
    ["label", "original_label", "final_prediction", "final_confidence", "file"]
]

In [ ]:
target_confidence = 0.72

# Set Matplotlib styling
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "grid.color": "lightgray",
        "grid.linestyle": "-",
    }
)

labels = final_predictions_by_dmr_groups_agg["label"]
predictions = final_predictions_by_dmr_groups_agg["final_prediction"]
confidence = final_predictions_by_dmr_groups_agg["final_confidence"]

confident_predictions = np.array(
    [int(x) if y > target_confidence else 39 for (x, y) in zip(predictions, confidence)]
)
# 1. Compute confusion matrix
cf_matrix = confusion_matrix(labels, confident_predictions)

# 2. Logic to exclude last row/col from Color Scaling
# We slice the matrix to get everything EXCEPT the last row and last column
subset_matrix = cf_matrix[:-1, :-1]
# Find the max value in the specific cell types
max_val_subset = np.max(subset_matrix)

# Create the heatmap
fig, ax = plt.subplots(1, figsize=(10, 10))

# 3. Apply the scaling using vmax
# 'vmax' clamps the color range. Anything higher than this (i.e., the last row/col)
# will appear as the darkest color (saturated), but won't distort the gradient for the rest.
cax = ax.matshow(cf_matrix, cmap="PuRd", vmin=0, vmax=max_val_subset)

# Add colorbar
# extend='max' indicates that values exist beyond the top of the colorbar
plt.colorbar(cax, fraction=0.046, pad=0.04, extend="max")

# 4. Annotate the heatmap
# We adjust the text color threshold.
# Since the last row/col is saturated, we force white text there for readability.
for (i, j), val in np.ndenumerate(cf_matrix):
    # Determine text color:
    # If we are in the "Background" zone (last row or col), usually dark color -> use White text
    # Else, use standard thresholding based on the subset max
    if i == cf_matrix.shape[0] - 1 or j == cf_matrix.shape[1] - 1:
        color = (
            "white" if val > (max_val_subset * 0.3) else "black"
        )  # Adjust contrast as needed
    else:
        color = "white" if val > (max_val_subset / 2) else "black"

    # Optional: If numbers are huge, use scientific notation or hide zeros
    label_text = f"{val}" if val > 0 else ""
    ax.text(j, i, label_text, ha="center", va="center", color=color, fontsize=8)

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=16, labelpad=20)
ax.set_ylabel("Ground-truth", fontsize=16, labelpad=20)

ax.set_xticks(range(len(labels_list)))
ax.set_xticklabels(labels_list, rotation=90, fontsize=8)  # Rotated for readability
ax.set_yticks(range(len(labels_list)))
ax.set_yticklabels(labels_list, fontsize=8)

# Move x-axis ticks to bottom (matshow puts them on top by default)
ax.xaxis.set_ticks_position("bottom")

# Set title
ax.set_title(
    f"MethylBERT: Attention Classifier: Regions aggregated resuts \n after selecting most confident prediction with minumum confidence {target_confidence}",
    fontsize=14,
    pad=20,
)

plt.tight_layout()
plt.show()

In [ ]:
test_data[test_data["total_methylated_labels"] > 3].head()

In [ ]:
test_data[test_data["read_id"] == 2961948].to_dict(orient="records")